# Experiment 1 · Notebook 08: Error analysis and the motivation for Experiment 2

*J26-IT-304 PP1. Baseline confusion classifier (TF-IDF + Logistic Regression). Section 14.*

Looks at **where the published model is wrong** (`models/clf-v1.0.0/`, loaded, never retrained) and asks the question that leads into PP2:
*how much of the error is about information the model never sees, namely the question the student was answering?* The last section,
**Motivation for Experiment 2**, is the main deliverable.

Section numbers run continuously across the Experiment 1 series, so a reference such as "EDA §3.5" or "section 8" points to the same place in every notebook:

| Notebook | Sections | Stage |
|---|---|---|
| `01_data_and_eda` | §1-§3 | provenance, load, exploratory analysis, leakage demonstration |
| `02_cleaning_and_labels` | §4-§5 | deduplication, group key, label mapping, proxy caveat |
| `03_split_and_baselines` | §6-§7 | group-aware split, trivial baselines, frozen OOD probe |
| `04_model_training_and_evaluation` | §8-§9 | TF-IDF + LR, group-aware CV, the single test evaluation |
| `05_features_and_ood_probe` | §10-§11 | coefficient inspection, stability, length, OOD probe evaluation |
| `06_sensitivity_check` | §12 | robustness of the label mapping: Partial Correct dropped |
| `07_publish_artifact` | §13 | the `clf-v1.0.0` artifact: pipeline, metadata, rules, verification |
| `08_error_analysis` | §14 | error analysis, blind judging, motivation for Experiment 2 |

Each notebook runs on its own from a clean kernel (kernel `exp1-confusion`). Its first code cell rebuilds its inputs with the same `confusion_baseline` functions the previous stage used, so the stages cannot drift apart. Run them in order 01 → 08 to regenerate every file in `reports/experiments/`.

**Licence rule for files.** The dataset's licence is unstated, so answer and question **text never goes into a committed file**. The committed CSVs
(`error_annotations.csv`, `error_summary.csv`) hold row ids, labels, judgements and numbers only; the sheets that contain text are written to
`data/processed/` (gitignored). Short excerpts are printed inside this notebook only, as in the earlier notebooks.

**Setup.** Imports and this stage's inputs: the **published artifact** (`pipeline.joblib`, `metadata.json`), the split rebuilt and proved identical by the
fingerprint stored in the metadata, and the loaded pipeline's predictions on the test set. The cell asserts that the metrics computed from those predictions
**equal `metadata["metrics"]` exactly**, so this analysis is of the model that is served. Nothing is retrained.

In [1]:
import json, re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

from confusion_baseline import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR, SEED
from confusion_baseline.data import load_raw
from confusion_baseline.errors import blind_sheet, control_ids, error_table
from confusion_baseline.evaluate import metrics
from confusion_baseline.prep import clean, map_labels
from confusion_baseline.split import group_split, split_fingerprint

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 100)
FIG_DIR = REPORTS_DIR / "experiments"
ART = MODELS_DIR / "clf-v1.0.0"

# ---- the published artifact (loaded, never refitted) ----
pipe = joblib.load(ART / "pipeline.joblib")
meta = json.loads((ART / "metadata.json").read_text(encoding="utf-8"))
assert list(pipe.classes_) == [0, 1] and meta["input"]["proba_index_confused"] == 1

# ---- the same split as notebooks 03-08 ----
df, followup_quarantine, _ = load_raw()
del followup_quarantine
y_primary = map_labels(clean(df)[0], "primary")
train, test = group_split(y_primary, test_size=0.2, seed=SEED)
fp = split_fingerprint(test)
assert fp == meta["split"]["test_fingerprint"], fp
print(f"PASS: test fingerprint equals metadata['split']['test_fingerprint'] ({fp[:12]}...); train {len(train)} rows, test {len(test)} rows")

# ---- predictions of the LOADED pipeline ----
proba = pipe.predict_proba(test["Answer"])[:, 1]
err = error_table(test, proba, threshold=meta["threshold"])
m = metrics(err["y"], err["pred"])
assert m == meta["metrics"], (m, meta["metrics"])
print(f"PASS: metrics of the loaded pipeline equal metadata['metrics'] exactly (F1(confused) {m['f1_confused']:.4f}, macro F1 {m['macro_f1']:.4f}) -- {meta['version']}")

n_fp, n_fn = int((err["error_type"] == "FP").sum()), int((err["error_type"] == "FN").sum())
assert (n_fp, n_fn, n_fp + n_fn) == (43, 24, 67), (n_fp, n_fn)
print(f"PASS: expected errors: FP = {n_fp} (understood called confused), FN = {n_fn} (confused called understood), total {n_fp + n_fn}")

PASS: test fingerprint equals metadata['split']['test_fingerprint'] (511fadfecbff...); train 860 rows, test 216 rows
PASS: metrics of the loaded pipeline equal metadata['metrics'] exactly (F1(confused) 0.7581, macro F1 0.6629) -- clf-v1.0.0
PASS: expected errors: FP = 43 (understood called confused), FN = 24 (confused called understood), total 67


## 14. Error analysis

### 14.1 The error table

**Why.** Every misclassified test row, with its gold 3-class label, gold binary label, prediction, P(confused), length and error type. The counts by error
type and gold Correctness show what kind of answers the model gets wrong. By construction, a **false negative (FN)** can only be a Partial Correct or Wrong answer
(gold confused, predicted understood) and a **false positive (FP)** can only be a Correct answer (gold understood, predicted confused).

In [2]:
errors = err[~err["correct"]].copy()
with pd.option_context("display.max_rows", None):
    display(errors[["row_id", "Correctness", "y", "pred", "p_confused", "tokens", "error_type"]]
            .rename(columns={"y": "gold (1 = confused)", "pred": "predicted"}).assign(p_confused=lambda d: d["p_confused"].round(3))
            .sort_values("row_id").reset_index(drop=True))
print("errors by error type x gold Correctness:")
display(pd.crosstab(errors["error_type"], errors["Correctness"], margins=True))
assert set(errors.loc[errors.error_type == "FN", "Correctness"]) <= {"Partial Correct", "Wrong"}
assert set(errors.loc[errors.error_type == "FP", "Correctness"]) == {"Correct"}
print("PASS: every FN is a Partial Correct or Wrong answer; every FP is a Correct answer")

,row_id,Correctness,gold (1 = confused),predicted,p_confused,tokens,error_type
0,0,Correct,0,1,0.855,24,FP
1,2,Wrong,1,0,0.173,22,FN
2,9,Correct,0,1,0.700,26,FP
3,16,Correct,0,1,0.992,22,FP
4,23,Correct,0,1,0.941,25,FP
5,24,Correct,0,1,0.959,20,FP
6,49,Partial Correct,1,0,0.454,11,FN
7,50,Wrong,1,0,0.475,12,FN
8,51,Correct,0,1,0.824,25,FP
9,133,Partial Correct,1,0,0.265,17,FN


errors by error type x gold Correctness:


Correctness,Correct,Partial Correct,Wrong,All
error_type,,,,
FN,0,10,14,24
FP,43,0,0,43
All,43,10,14,67


PASS: every FN is a Partial Correct or Wrong answer; every FP is a Correct answer


### 14.2 Category definitions (written before any labelling)

**Why.** Defining the categories first stops the definitions drifting to fit the results. They can **overlap** (a row can have several), as in the PP1 plan section 4.9.

| Category | How it is decided | Definition |
|---|---|---|
| `short` | **computed** | token length <= the 25th percentile of the **train** answer lengths (the cut-off is printed below) |
| `partially_correct` | **computed** | gold `Correctness == "Partial Correct"` |
| `confident_but_incorrect` | **computed** | gold `Wrong` **and** predicted understood. The EDA found 0 hedging words in 1,078 answers, so every Wrong answer is confidently phrased; this counts the ones that fooled the model |
| `needs_context` | **judged** | a reader who sees **only the answer, not the question**, cannot tell whether it is correct (blind pass 1 = `cannot_tell`) |
| `ambiguous` | **judged** | even **with the question**, the gold label is debatable (blind pass 2 disagrees with the gold label, or is still `cannot_tell`): label noise |

**Primary category** (one per row, so the shares add to 100%), by this precedence: `needs_context` > `ambiguous` > `partially_correct` >
`confident_but_incorrect` > `short` > `other`. The multi-label counts are the main table; the primary category is a convenience.

In [3]:
train_tokens = train["Answer"].str.split().str.len()
SHORT_CUTOFF = float(np.percentile(train_tokens, 25))
print(f"25th percentile of TRAIN answer length = {SHORT_CUTOFF:.1f} tokens  ->  short means tokens <= {SHORT_CUTOFF:.1f}")
print(f"(train: median {train_tokens.median():.0f}, min {train_tokens.min()}, max {train_tokens.max()}; test errors: median {errors['tokens'].median():.0f})")

25th percentile of TRAIN answer length = 15.0 tokens  ->  short means tokens <= 15.0
(train: median 19, min 8, max 36; test errors: median 20)


### 14.3 Blind judging protocol

**Why. The control sample is what makes the context number defensible.** The model never sees the question (by design: the question is the conversational
context that PP2 adds), so many *correctly* classified answers may also be impossible to judge without it. "X% of errors need context" means nothing on its own; the finding is
the **difference** between the error rate and the control rate. So **67 correctly classified test rows** (seed 42) are judged exactly like the 67 errors.

**Protocol.**
1. Errors and controls are **shuffled together** (seed 42) under opaque item ids `B001`...`B134`. The pass-1 sheet shows **only the item id and the answer text**: no gold
   label, no prediction, no question, no set. It is written to `data/processed/blind_pass1.csv` (gitignored).
2. **Pass 1 (answer only).** For each item: `understood` (the answer reads as correct), `confused` (it contains a clear error or is clearly incomplete) or `cannot_tell`
   (correctness depends on something not in the answer: a yes/no whose question is unseen, "the code" or "the condition" with no code). `needs_context` = `cannot_tell`.
   **All 134 items were judged in pass 1 before any question was looked at.**
3. **Pass 2 (with the question)**, still with no gold label and no prediction: the same three values. `ambiguous` = pass 2 disagrees with the gold label, or is still `cannot_tell`.
4. Only after both passes were complete were the judgements joined to row ids and sets.

**Disclosure.** The judgements were made by **an AI assistant, as a single annotator**. They are **not made by a human expert and are not validated** (no second annotator, no
agreement statistic). That is a limitation of this notebook, and a reason PP2 needs human annotation. A different annotator could draw the `understood` / `confused` line
differently, in particular for terse partial answers.

In [4]:
ctrl = control_ids(err, len(errors), seed=SEED)
sheet = blind_sheet(test, sorted(errors["row_id"]), ctrl, seed=SEED)
assert len(sheet) == 134 and sheet["row_id"].is_unique
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
sheet[["item_id", "Answer"]].to_csv(DATA_PROCESSED / "blind_pass1.csv", index=False)               # shown in pass 1: id + answer ONLY
sheet[["item_id", "Question", "Answer"]].to_csv(DATA_PROCESSED / "blind_pass2.csv", index=False)   # shown in pass 2: + question
sheet[["item_id", "row_id", "set"]].to_csv(DATA_PROCESSED / "blind_key.csv", index=False)           # key, joined only after judging
print("pass-1 sheet columns:", list(pd.read_csv(DATA_PROCESSED / "blind_pass1.csv").columns), "(no label, prediction, question or set)")
print("written to data/processed/ (gitignored): blind_pass1.csv, blind_pass2.csv, blind_key.csv")

ann = pd.read_csv(FIG_DIR / "error_annotations.csv")
print("error_annotations.csv columns:", list(ann.columns), "| rows:", len(ann))
chk = sheet[["item_id", "row_id", "set"]].merge(ann[["item_id", "row_id", "set"]], on="item_id", suffixes=("", "_ann"))
assert len(chk) == 134 and (chk["row_id"] == chk["row_id_ann"]).all() and (chk["set"] == chk["set_ann"]).all()
print("PASS: the annotations' item ids, row ids and sets match the seeded blinded sheet rebuilt here")
assert set(ann.loc[ann["set"] == "error", "row_id"]) == set(errors["row_id"])
assert set(ann.loc[ann["set"] == "control", "row_id"]) == set(ctrl) and not set(ctrl) & set(errors["row_id"])
print(f"PASS: 'error' items are exactly the {len(errors)} misclassified rows; 'control' items are {len(ctrl)} correctly classified rows (seed {SEED})")
print("\njudgements (counts):")
display(pd.concat([ann.groupby("set")["pass1"].value_counts().unstack(fill_value=0).add_prefix("pass1: "),
                   ann.groupby("set")["pass2"].value_counts().unstack(fill_value=0).add_prefix("pass2: ")], axis=1))

pass-1 sheet columns: ['item_id', 'Answer'] (no label, prediction, question or set)
written to data/processed/ (gitignored): blind_pass1.csv, blind_pass2.csv, blind_key.csv
error_annotations.csv columns: ['item_id', 'row_id', 'set', 'pass1', 'pass2', 'reason1', 'reason2'] | rows: 134
PASS: the annotations' item ids, row ids and sets match the seeded blinded sheet rebuilt here
PASS: 'error' items are exactly the 67 misclassified rows; 'control' items are 67 correctly classified rows (seed 42)

judgements (counts):


,pass1: cannot_tell,pass1: confused,pass1: understood,pass2: cannot_tell,pass2: confused,pass2: understood
set,,,,,,
control,2,37,28,1,39,27
error,8,19,40,3,22,42


### 14.4 Results

**Why.** All five categories are assembled per item (computed ones from the data, judged ones from the annotation file, which is **fixed**, so the output is reproducible),
then counted for the errors (all, FP only, FN only) and for the control sample. `confident_but_incorrect` is an error-only category by definition (a control row
is predicted correctly), so it is 0 for controls.

In [5]:
A = ann.merge(err[["row_id", "Correctness", "y", "pred", "p_confused", "tokens", "error_type"]], on="row_id")
A["group"] = np.where(A["set"] == "control", "control", A["error_type"])
gold = np.where(A["y"] == 1, "confused", "understood")
A["short"] = A["tokens"] <= SHORT_CUTOFF
A["partially_correct"] = A["Correctness"] == "Partial Correct"
A["confident_but_incorrect"] = (A["Correctness"] == "Wrong") & (A["pred"] == 0)
A["needs_context"] = A["pass1"] == "cannot_tell"
A["ambiguous"] = (A["pass2"] == "cannot_tell") | ((A["pass2"] != "cannot_tell") & (A["pass2"] != gold))
CATS = ["short", "partially_correct", "confident_but_incorrect", "needs_context", "ambiguous"]
PRECEDENCE = ["needs_context", "ambiguous", "partially_correct", "confident_but_incorrect", "short"]
A["primary"] = "other"
for c in reversed(PRECEDENCE):
    A.loc[A[c], "primary"] = c

SETS = {"errors (all)": A["set"] == "error", "FP": A["group"] == "FP", "FN": A["group"] == "FN", "control": A["set"] == "control"}
sizes = {k: int(v.sum()) for k, v in SETS.items()}
print("group sizes:", sizes)

def table(rows, label, primary=False):
    out = {}
    for k, mask in SETS.items():
        flags = (lambda r: A.loc[mask, "primary"] == r) if primary else (lambda r: A.loc[mask, r])
        out[k] = [f"{int(flags(r).sum())} ({flags(r).mean():.0%})" for r in rows]
    t = pd.DataFrame(out, index=rows)
    t.index.name = label
    return t

print("MULTI-LABEL counts (a row can be in several categories); n (share of the group):")
display(table(CATS, "category"))
print("PRIMARY category (one per row, precedence needs_context > ambiguous > partially_correct > confident_but_incorrect > short > other):")
display(table(PRECEDENCE + ["other"], "primary category", primary=True))
assert (A.groupby("set")["primary"].count() == A.groupby("set").size()).all()

group sizes: {'errors (all)': 67, 'FP': 43, 'FN': 24, 'control': 67}
MULTI-LABEL counts (a row can be in several categories); n (share of the group):


,errors (all),FP,FN,control
category,,,,
short,11 (16%),3 (7%),8 (33%),15 (22%)
partially_correct,10 (15%),0 (0%),10 (42%),24 (36%)
confident_but_incorrect,14 (21%),0 (0%),14 (58%),0 (0%)
needs_context,8 (12%),5 (12%),3 (12%),2 (3%)
ambiguous,9 (13%),5 (12%),4 (17%),12 (18%)


PRIMARY category (one per row, precedence needs_context > ambiguous > partially_correct > confident_but_incorrect > short > other):


,errors (all),FP,FN,control
primary category,,,,
needs_context,8 (12%),5 (12%),3 (12%),2 (3%)
ambiguous,8 (12%),4 (9%),4 (17%),11 (16%)
partially_correct,6 (9%),0 (0%),6 (25%),15 (22%)
confident_but_incorrect,11 (16%),0 (0%),11 (46%),0 (0%)
short,3 (4%),3 (7%),0 (0%),8 (12%)
other,31 (46%),31 (72%),0 (0%),31 (46%)


**THE KEY TABLE.** `needs_context` is the share of items whose correctness could **not** be judged from the answer alone (pass 1 = `cannot_tell`). It is compared
between the errors and the controls with 95% Wilson intervals, the difference with a Newcombe 95% interval (a method for two independent proportions) and Fisher's exact test.
The same is done for `ambiguous`.

In [6]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half

def newcombe(k1, n1, k2, n2):
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson(k1, n1)
    l2, u2 = wilson(k2, n2)
    d = p1 - p2
    return d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)

def compare(col, mask_a, mask_b, name_a="errors", name_b="controls"):
    ka, na, kb, nb = int(A.loc[mask_a, col].sum()), int(mask_a.sum()), int(A.loc[mask_b, col].sum()), int(mask_b.sum())
    la, ua = wilson(ka, na)
    lb, ub = wilson(kb, nb)
    dl, du = newcombe(ka, na, kb, nb)
    p = fisher_exact([[ka, na - ka], [kb, nb - kb]])[1]
    return {"category": col, f"{name_a}": f"{ka}/{na} = {ka / na:.1%}", f"{name_a} 95% CI": f"[{la:.1%}, {ua:.1%}]",
            f"{name_b}": f"{kb}/{nb} = {kb / nb:.1%}", f"{name_b} 95% CI": f"[{lb:.1%}, {ub:.1%}]",
            "difference (pp)": f"{100 * (ka / na - kb / nb):+.1f}", "difference 95% CI (pp)": f"[{100 * dl:+.1f}, {100 * du:+.1f}]",
            "Fisher p": f"{p:.3f}"}

E, C = SETS["errors (all)"], SETS["control"]
key = pd.DataFrame([compare("needs_context", E, C), compare("ambiguous", E, C)]).set_index("category")
print("THE KEY TABLE: errors vs correctly classified controls")
display(key.T)
print("by error type, for needs_context:")
by_type = []
for grp in ["FP", "FN"]:
    r = compare("needs_context", SETS[grp], C, "errors of this type", "controls")
    r["category"] = f"needs_context: {grp} vs controls"
    by_type.append(r)
display(pd.DataFrame(by_type).set_index("category").T)

THE KEY TABLE: errors vs correctly classified controls


category,needs_context,ambiguous
errors,8/67 = 11.9%,9/67 = 13.4%
errors 95% CI,"[6.2%, 21.8%]","[7.2%, 23.6%]"
controls,2/67 = 3.0%,12/67 = 17.9%
controls 95% CI,"[0.8%, 10.2%]","[10.6%, 28.7%]"
difference (pp),+9.0,-4.5
difference 95% CI (pp),"[-0.3, +19.1]","[-17.0, +8.1]"
Fisher p,0.096,0.635


by error type, for needs_context:


category,needs_context: FP vs controls,needs_context: FN vs controls
errors of this type,5/43 = 11.6%,3/24 = 12.5%
errors of this type 95% CI,"[5.1%, 24.5%]","[4.3%, 31.0%]"
controls,2/67 = 3.0%,2/67 = 3.0%
controls 95% CI,"[0.8%, 10.2%]","[0.8%, 10.2%]"
difference (pp),+8.6,+9.5
difference 95% CI (pp),"[-1.1, +21.7]","[-1.4, +28.1]"
Fisher p,0.108,0.112


**Pass-1 accuracy.** Among the items where an answer-only reading *could* decide (pass 1 is `understood` or `confused`), how often does that reading agree with the gold
label? It shows how far an answer alone can be judged at all, and whether the errors are the answers that are hard to judge **even for an annotator who sees the answer**.

In [7]:
rows = []
for name, mask in [("errors", E), ("controls", C)]:
    sub = A[mask & (A["pass1"] != "cannot_tell")]
    k = int((sub["pass1"] == np.where(sub["y"] == 1, "confused", "understood")).sum())
    lo, hi = wilson(k, len(sub))
    rows.append({"set": name, "items judgeable from the answer": f"{len(sub)} of {int(mask.sum())}",
                 "pass-1 agrees with gold": f"{k}/{len(sub)} = {k / len(sub):.1%}", "95% CI": f"[{lo:.1%}, {hi:.1%}]"})
display(pd.DataFrame(rows).set_index("set"))
sub = A[A["pass2"] != "cannot_tell"]
for name, mask in [("errors", E), ("controls", C)]:
    s2 = sub[mask.loc[sub.index]]
    print(f"pass 2 (with the question) agrees with gold, {name}: {int((s2['pass2'] == np.where(s2['y'] == 1, 'confused', 'understood')).sum())}/{len(s2)}")

,items judgeable from the answer,pass-1 agrees with gold,95% CI
set,,,
errors,59 of 67,53/59 = 89.8%,"[79.5%, 95.3%]"
controls,65 of 67,53/65 = 81.5%,"[70.4%, 89.1%]"


pass 2 (with the question) agrees with gold, errors: 58/64
pass 2 (with the question) agrees with gold, controls: 55/66


**Error rate by length.** Test answers are cut into length quartiles. Notebook 05 found the model's score barely tracks length and that it answers "confused" to short, unfamiliar
text, so the error rate and the FP / FN mix per bucket show whether the mistakes concentrate at one end.

In [8]:
test_len = test["Answer"].str.split().str.len().to_numpy()
bucket = pd.qcut(pd.Series(test_len), 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"])
lb = pd.DataFrame({"bucket": bucket.to_numpy(), "tokens": test_len, "error": ~err["correct"].to_numpy(),
                   "FP": (err["error_type"] == "FP").to_numpy(), "FN": (err["error_type"] == "FN").to_numpy(), "confused": err["y"].to_numpy() == 1})
g = lb.groupby("bucket", observed=True).agg(tokens_min=("tokens", "min"), tokens_max=("tokens", "max"), rows=("error", "size"),
                                              errors=("error", "sum"), FP=("FP", "sum"), FN=("FN", "sum"), gold_confused_share=("confused", "mean"))
g["error rate"] = (g["errors"] / g["rows"]).round(3)
g["gold_confused_share"] = g["gold_confused_share"].round(3)
display(g)
print("mean P(confused) and length by outcome:")
display(err.assign(outcome=np.select([(err.y == 1) & (err.pred == 1), (err.y == 0) & (err.pred == 0), err.error_type == "FP", err.error_type == "FN"], ["TP", "TN", "FP", "FN"], default="?"))
        .groupby("outcome").agg(n=("row_id", "size"), mean_tokens=("tokens", "mean"), mean_p_confused=("p_confused", "mean")).round(3))

,tokens_min,tokens_max,rows,errors,FP,FN,gold_confused_share,error rate
bucket,,,,,,,,
Q1 shortest,7,15,58,11,3,8,0.810,0.190
Q2,16,18,52,15,8,7,0.692,0.288
Q3,19,22,59,23,15,8,0.525,0.390
Q4 longest,23,32,47,18,17,1,0.319,0.383


mean P(confused) and length by outcome:


,n,mean_tokens,mean_p_confused
outcome,,,
FN,24,16.833,0.261
FP,43,20.698,0.772
TN,44,20.659,0.234
TP,105,17.019,0.825


**Examples.** Up to three printed excerpts (seed 42) per category, taken from the **errors**, so the categories can be read, not just counted. Excerpts are short and
appear in this notebook only.

In [9]:
test_i = test.set_index("row_id")
def show(cat, k=3):
    sub = A[(A["set"] == "error") & A[cat]].sort_values("row_id")
    pick = sub.sample(min(k, len(sub)), random_state=SEED) if len(sub) else sub
    print(f"=== {cat}  ({len(sub)} errors; {len(pick)} shown)")
    for _, r in pick.iterrows():
        print(f"  [row {r.row_id}] gold {r.Correctness} ({'confused' if r.y else 'understood'}), predicted {'confused' if r.pred else 'understood'}, P(confused) {r.p_confused:.2f}, {r.tokens} tokens"
              f", pass1={r.pass1}, pass2={r.pass2}")
        print(f"      Q: {test_i.loc[r.row_id, 'Question'][:120]}")
        print(f"      A: {test_i.loc[r.row_id, 'Answer'][:200]}")
for c in CATS:
    show(c)

=== short  (11 errors; 3 shown)
  [row 699] gold Wrong (confused), predicted understood, P(confused) 0.25, 11 tokens, pass1=confused, pass2=confused
      Q: What is the primary difference between Java and C regarding memory management?
      A: Java uses manual memory management, whereas C uses a garbage collector.
  [row 49] gold Partial Correct (confused), predicted understood, P(confused) 0.45, 11 tokens, pass1=confused, pass2=confused
      Q: What is the `this` keyword in Java?
      A: `this` is used to refer to the current class in Java.
  [row 907] gold Wrong (confused), predicted understood, P(confused) 0.38, 13 tokens, pass1=cannot_tell, pass2=confused
      Q: What is the value of the ternary operator `true ? 1 : 0`?
      A: The value will be `0`, as `false` is returned by the ternary operator.
=== partially_correct  (10 errors; 3 shown)
  [row 1009] gold Partial Correct (confused), predicted understood, P(confused) 0.41, 24 tokens, pass1=cannot_tell, pass2=confused
      

**Reading section 14.4.**

- **What the errors are.** Of the 67 errors, 43 are FP (correct answers called confused) and 24 are FN (14 Wrong, 10 Partial Correct called understood). **14 of the 24 FN (58%) are
  `confident_but_incorrect`**: Wrong answers the model passed as understood, for example "`compareTo()` belongs to the `Object` class" or "an interface cannot contain any methods". A
  lexical model cannot see that these are false statements. **31 of the 43 FP (72%) fall in no category at all**: ordinary correct answers that the model called confused.
- **Short answers are not where the model fails.** `short` (15 tokens or fewer, the train 25th percentile) is 16% of errors but 22% of the controls, so short answers are *not* over-represented among the mistakes.
- **Judged categories.** `needs_context` is **8 of 67 errors (12%)** against **2 of 67 controls (3%)**; `ambiguous` (label debatable even with the question) is 9 errors (13%) against 12 controls (18%).
  The items judged `cannot_tell` from the answer alone are mostly yes/no answers whose question is not shown, and answers about "the code" or "the condition" (see the excerpts), as in `if (x == 5)` and the value of a ternary expression.
- **The errors are mostly judgeable from the answer alone.** For 59 of the 67 errors an answer-only reading could decide, and it **agrees with the gold label in 53 of them (89.8%, 95% CI 79.5% to 95.3%)**, compared with 53 of 65 (81.5%)
  for the controls. With the question shown, agreement with gold is 58 of 64 for errors and 55 of 66 for controls. So the model's errors are **not** concentrated in labels that are noisy (errors are *not* more `ambiguous` than controls)
  and not in answers that cannot be judged without context: they are answers that an annotator reading only the answer would have classified as the gold label does.
- **Length.** The error rate is 19% in the shortest quartile (7-15 tokens) and 39% / 38% in the two longest. FP grows with length (3, 8, 15, 17 per quartile) while FN does not (8, 7, 8, 1), while the
  gold confused share falls from 81% to 32%. Long correct answers are the ones the model calls confused. This fits notebooks 04 and 05: TF-IDF is length-normalised and blind to the length signal a one-feature baseline
  uses, and its score barely tracks length (corr -0.15).

### 14.5 False positives vs false negatives

**Why.** The two errors have different costs in the tutor, so which one dominates matters. The counts, rates and the length of each outcome are printed first.

In [10]:
n_u, n_c = int((err["y"] == 0).sum()), int((err["y"] == 1).sum())
print(f"FP = {n_fp} of {n_u} understood answers = {n_fp / n_u:.1%} of understood answers are called confused")
print(f"FN = {n_fn} of {n_c} confused answers = {n_fn / n_c:.1%} of confused answers are called understood")
print(f"FP : FN = {n_fp / n_fn:.2f} : 1   |   threshold {meta['threshold']}, class_weight = {meta['hyperparameters']['logistic_regression']['class_weight']!r}")
print("FN by gold class:", errors[errors.error_type == 'FN']['Correctness'].value_counts().to_dict())

FP = 43 of 87 understood answers = 49.4% of understood answers are called confused
FN = 24 of 129 confused answers = 18.6% of confused answers are called understood
FP : FN = 1.79 : 1   |   threshold 0.5, class_weight = 'balanced'
FN by gold class: {'Wrong': 14, 'Partial Correct': 10}


**FP vs FN, and their costs.**

- **FN = the tutor moves on while the student is lost.** For a tutor this is the **worse error pedagogically**: the student gets no extra help at the moment it is needed, and the gap goes unnoticed.
- **FP = an unnecessary strategy escalation.** The tutor re-explains or simplifies something the student already understood, which wastes the student's time, and the weakness logger then counts the event toward a **weak topic** that the student does not actually have.

**Which dominates for this model.** By count, **FP dominates: 43 FP against 24 FN (1.79 : 1)** at threshold 0.5. That is 49% of understood answers called confused, against 19% of confused answers called understood (understood recall 0.506 and confused recall 0.814, as in
notebook 04). The FPs are not borderline: their mean P(confused) is 0.77 against 0.23 for the correctly classified understood answers. Likely reasons: (1) the lexical features recognise *wrong* answers better than *right* ones, so many correct answers
get no signal that says "understood"; (2) notebook 05 showed that the model's default for text it does not recognise is "confused" (intercept +1.45, P = 0.81 for an answer with no known word), the same short-text / unfamiliar-text bias that appears
on the probe; (3) `class_weight='balanced'`, which up-weights the minority understood class, did not prevent it. Whether FP or FN **costs more in total** depends on how much worse one FN is than one FP, which is a product decision, not something this data
can settle; no threshold was tuned here (the threshold stays at 0.5), and the FN side should not be neglected simply because it is the smaller count.

### 14.6 The summary file

**Why.** `error_summary.csv` holds the category x group counts and shares (multi-label and primary), with **no text**, so the numbers can be committed and cited.

In [11]:
rows = []
for kind, cats in [("multi-label", CATS), ("primary", PRECEDENCE + ["other"])]:
    for grp, mask in SETS.items():
        n = int(mask.sum())
        for c in cats:
            k = int(A.loc[mask, c].sum()) if kind == "multi-label" else int((A.loc[mask, "primary"] == c).sum())
            rows.append({"kind": kind, "group": grp, "category": c, "n": k, "group_size": n, "share": round(k / n, 4)})
summary = pd.DataFrame(rows)
summary.to_csv(FIG_DIR / "error_summary.csv", index=False)
print("saved: reports/experiments/error_summary.csv", summary.shape, "| columns:", list(summary.columns))
print("error_annotations.csv columns:", list(ann.columns))
text_cols = {"Question", "Answer", "text"}
assert not text_cols & set(summary.columns) and not text_cols & set(ann.columns)
print("PASS: neither committed CSV has a question or answer text column")

saved: reports/experiments/error_summary.csv (44, 6) | columns: ['kind', 'group', 'category', 'n', 'group_size', 'share']
error_annotations.csv columns: ['item_id', 'row_id', 'set', 'pass1', 'pass2', 'reason1', 'reason2']
PASS: neither committed CSV has a question or answer text column


## Motivation for Experiment 2

**The number.** On the 67 errors of the published model, **11.9% (8 of 67; 95% CI 6.2% to 21.8%) could not be judged from the answer alone, against 3.0% (2 of 67; 0.8% to 10.2%) of the 67 correctly classified controls.**
The difference is **+9.0 percentage points (95% CI -0.3 to +19.1, Fisher p = 0.096)**. The control matters: the model never sees the question, so some correct predictions also "need context", and 3% is the background rate. The difference is in the direction the PP2
hypothesis predicts, and about four times the background, **but it is not clearly different from zero** with 67 items per group, and the absolute level is small. **Ambiguity (label noise) is not higher among errors**: 13.4% vs 17.9% (difference -4.5 pp, CI -17.0 to +8.1).

**What it does not show.** **Y is not clearly higher than Z.** On this data, missing context is **not shown to be the main failure mode**. 88% of the errors (59 of 67) could be judged from the answer alone, and an answer-only reading agrees with the gold label for 89.8% of those. These errors
are mostly answers that an annotator who sees only the answer would classify as the gold label does, and the model gets them wrong: confidently false statements it passes as understood, and correct answers it flags as confused. That is a failure of
**judging the content of an answer**, which a vocabulary model cannot do, and not mainly a failure of missing context. Asserting the context hypothesis from these numbers would be wrong.

**Be honest about the design.** The question is precisely the context the model lacks, so this measures how much *the missing preceding turn* matters on **this proxy task** (is a single synthetic answer correct?). It is not evidence about real multi-turn tutoring, where the
preceding tutor turn plays a different role. The judgements were made by **an AI assistant as a single annotator, not validated by a human expert**, with 67 items per group, so the intervals are wide. A human annotator could shift the `cannot_tell` counts in either direction.

**How this ties to notebooks 05 and 06.**
- **The lexical model does not beat answer length** (notebooks 04 and 06): the vocabulary signal is weak and partly generator style, and the errors here are content judgements it cannot make.
- **The probe shows the proxy gap** (notebook 05): on 40 hand-written student-style messages the model flags 34 of 40 as confused, with specificity 0.20 and AUC 0.57. Real student turns such as "I don't get it" or "that makes sense" are mostly *not answers*, and what they refer to is
  the preceding tutor turn. That is the setting where context could matter, and this dataset (answers to isolated questions) contains none of it.
- **The Partial boundary is not where the difficulty sits** (notebook 06): dropping Partial Correct does not make the task easier for the lexical model, and here only 10 of the 67 errors are Partial answers.

**PP2's question, restated.** *Does conversational context (the preceding tutor turn or turns) improve the detection of real student confusion, compared with the student's message alone?* This notebook argues that the question has to be tested on **different data from this dataset**, because on a
synthetic answer-correctness proxy the missing question is not what limits a lexical baseline. PP2 needs:
1. **Real student turns**, not generated answers, including the many that contain no answer (explicit confusion, hedged attempts, explicit understanding).
2. **The preceding tutor turn(s)** stored with every student turn, so that the same message can be evaluated **with and without** its context.
3. **Human labels** for confusion (not correctness), ideally by **two annotators** with a reported agreement, so that label noise and the single-annotator limit of this notebook are measured.
4. **The design used here as the template:** a blind pass on the message alone, then a pass with context, and a **control sample** of correctly handled turns, so that "needs context" is always compared against its background rate.

## Findings of notebook 08

| Topic | Finding | Where |
|---|---|---|
| Setup | Loads the **published** `clf-v1.0.0` (never retrained); the metrics of the loaded pipeline equal `metadata["metrics"]` exactly. 67 errors = **43 FP + 24 FN** (FN: 14 Wrong, 10 Partial Correct). | Setup, §14.1 |
| Blind judging | 67 errors + 67 correctly classified controls (seed 42), shuffled, answer-only pass then with-question pass, no gold or prediction shown. **Judged by an AI assistant as a single annotator, not validated**; a limitation and a reason PP2 needs human annotation. | §14.3 |
| What the errors are | 14 of 24 FN (58%) are `confident_but_incorrect`; 31 of 43 FP (72%) fit no category (plain correct answers called confused). `short` answers are **not** over-represented (16% of errors vs 22% of controls). | §14.4 |
| **Needs context** | **11.9% of errors** (6.2%-21.8%) vs **3.0% of controls** (0.8%-10.2%): difference **+9.0 pp, 95% CI -0.3 to +19.1, p = 0.096**. Direction consistent with the hypothesis; **not clearly different from zero**; small in absolute terms. | §14.4 |
| Ambiguous | Label noise is **not** higher among errors: 13.4% vs 17.9% (-4.5 pp, CI -17.0 to +8.1). | §14.4 |
| Judgeable from the answer | 59 of 67 errors could be judged from the answer alone, and the answer-only reading agrees with the gold label in **89.8%** (79.5%-95.3%) of them (controls 81.5%). The errors are mostly content judgements a lexical model cannot make, not missing context. | §14.4 |
| Length | Error rate 19% in the shortest quartile vs 39% / 38% in the longest; FP rises with length (3, 8, 15, 17), FN does not. Long correct answers are called confused (TF-IDF is blind to length). | §14.4 |
| FP vs FN | FP dominates by count (**43 vs 24**, 1.79 : 1); FN is the worse error per event. FPs are confident (mean P 0.77), so this is not a threshold effect; no threshold was tuned. | §14.5 |
| **Motivation for Experiment 2** | Y is **not clearly higher** than Z: context is **not shown to be the main failure mode on this proxy task**. PP2 must be tested on real student turns with the preceding tutor turn and human labels (two annotators), using this blind-pass-plus-control design. | Motivation |
| Files | `error_annotations.csv` and `error_summary.csv` hold ids, labels, judgements and numbers only (no question or answer text); the blind sheets are in gitignored `data/processed/`. | §14.3, §14.6 |